# CommercePulse — Análise de Vendedores com Concentração de Atrasos

Este notebook investiga quais vendedores concentram a maior parte dos atrasos de entrega,
aplicando o **Princípio de Pareto (80/20)** para identificar oportunidades de melhoria
logística com ações direcionadas.

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Configuração de caminhos
BASE_DIR = Path().resolve().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Carregar base processada
df = pd.read_csv(PROCESSED_DIR / "commercepulse_orders_items.csv")

date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Base carregada: {df.shape[0]:,} itens, {df['order_id'].nunique():,} pedidos")

## 1. Filtrar Pedidos Entregues

In [ ]:
delivered = df[(df["is_delivered"] == 1) & (df["is_late"].notna())].copy()

print(f"Itens entregues: {len(delivered):,}")
print(f"Taxa de atraso global: {delivered['is_late'].mean()*100:.1f}%")
print(f"Total de itens atrasados: {int(delivered['is_late'].sum()):,}")

## 2. Métricas por Vendedor

In [ ]:
seller_metrics = delivered.groupby("seller_id").agg(
    total_items=("order_item_id", "count"),
    total_orders=("order_id", "nunique"),
    total_revenue=("item_revenue", "sum"),
    avg_review=("review_score", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
    delay_rate=("is_late", "mean"),
    late_items=("is_late", "sum"),
    avg_delay=("delay_days", "mean"),
    seller_state=("seller_state", "first"),
    seller_city=("seller_city", "first"),
).reset_index()

print(f"Total de vendedores: {len(seller_metrics):,}")
print(f"Vendedores com atrasos: {(seller_metrics['late_items'] > 0).sum():,}")
seller_metrics.describe()

## 3. Distribuição da Taxa de Atraso por Vendedor

In [ ]:
fig = px.histogram(
    seller_metrics[seller_metrics["total_orders"] >= 10],
    x="delay_rate",
    nbins=50,
    color_discrete_sequence=["#636EFA"],
    labels={"delay_rate": "Taxa de Atraso", "count": "Qtd Vendedores"},
    title="Distribuição da Taxa de Atraso por Vendedor (mín. 10 pedidos)",
    template="plotly_dark",
    height=400,
)
fig.update_layout(xaxis_tickformat=".0%")
fig.show()

## 4. Top 20 Vendedores com Maiores Taxas de Atraso

Filtramos vendedores com volume mínimo de 30 pedidos para evitar ruído estatístico.

In [ ]:
min_orders = 30
sellers_vol = seller_metrics[seller_metrics["total_orders"] >= min_orders].copy()
sellers_vol = sellers_vol.sort_values("delay_rate", ascending=False)

top20 = sellers_vol.head(20)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top20["seller_id"].str[:8] + "...",
    y=top20["delay_rate"] * 100,
    marker=dict(
        color=top20["delay_rate"] * 100,
        colorscale="Reds",
        showscale=True,
        colorbar=dict(title="Taxa de<br>Atraso (%)"),
    ),
    text=top20["delay_rate"].apply(lambda x: f"{x*100:.1f}%"),
    textposition="outside",
))
fig.update_layout(
    title=f"Top 20 Vendedores com Maiores Taxas de Atraso (mín. {min_orders} pedidos)",
    template="plotly_dark", height=500,
    xaxis_title="Vendedor (ID parcial)", yaxis_title="Taxa de Atraso (%)",
    xaxis_tickangle=-45,
)
fig.show()

## 5. Análise de Pareto — Concentração de Atrasos

O princípio de Pareto sugere que **20% das causas geram 80% dos efeitos**.
Vamos verificar se poucos vendedores são responsáveis pela maioria dos atrasos.

In [ ]:
pareto = seller_metrics[seller_metrics["late_items"] > 0].sort_values("late_items", ascending=False).reset_index(drop=True)
pareto["cumulative_late"] = pareto["late_items"].cumsum()
pareto["cumulative_pct"] = pareto["cumulative_late"] / pareto["late_items"].sum() * 100

n_80 = int((pareto["cumulative_pct"] <= 80).sum()) + 1
pct_sellers_80 = n_80 / len(seller_metrics) * 100

print(f"Total de vendedores com atrasos: {len(pareto):,}")
print(f"Vendedores responsáveis por 80% dos atrasos: {n_80} ({pct_sellers_80:.1f}% do total)")

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=list(range(1, len(pareto) + 1)),
        y=pareto["late_items"],
        name="Itens Atrasados",
        marker_color="#EF553B",
        opacity=0.7,
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=list(range(1, len(pareto) + 1)),
        y=pareto["cumulative_pct"],
        name="% Acumulado",
        line=dict(color="#00CC96", width=2.5),
    ),
    secondary_y=True,
)
fig.add_hline(y=80, line_dash="dash", line_color="yellow", opacity=0.5, secondary_y=True)

fig.update_layout(
    title=f"Pareto de Atrasos: {n_80} vendedores ({pct_sellers_80:.0f}%) concentram 80% dos atrasos",
    template="plotly_dark", height=500,
    xaxis_title="Vendedores (ordenados por atrasos)",
    legend=dict(x=0.65, y=0.3),
)
fig.update_yaxes(title_text="Itens Atrasados", secondary_y=False)
fig.update_yaxes(title_text="% Acumulado", secondary_y=True)
fig.show()

## 6. Correlação: Taxa de Atraso vs. Nota Média

Vendedores com altas taxas de atraso tendem a ter notas piores?

In [ ]:
fig = px.scatter(
    sellers_vol,
    x="delay_rate",
    y="avg_review",
    size="total_orders",
    color="seller_state",
    hover_data=["seller_id", "total_orders", "total_revenue"],
    labels={
        "delay_rate": "Taxa de Atraso",
        "avg_review": "Nota Média",
        "total_orders": "Total de Pedidos",
        "seller_state": "Estado",
    },
    title="Taxa de Atraso vs. Nota Média por Vendedor",
    template="plotly_dark",
    height=550,
)
fig.update_layout(xaxis_tickformat=".0%")
fig.show()

## 7. Taxa de Atraso por Estado do Vendedor

In [ ]:
state_seller = delivered.groupby("seller_state").agg(
    total_sellers=("seller_id", "nunique"),
    avg_delay_rate=("is_late", "mean"),
    total_orders=("order_id", "nunique"),
    avg_review=("review_score", "mean"),
).reset_index().sort_values("avg_delay_rate", ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=state_seller["seller_state"],
    y=state_seller["avg_delay_rate"] * 100,
    marker=dict(
        color=state_seller["avg_delay_rate"] * 100,
        colorscale="YlOrRd",
    ),
    text=state_seller["avg_delay_rate"].apply(lambda x: f"{x*100:.1f}%"),
    textposition="outside",
))
fig.update_layout(
    title="Taxa de Atraso Média por Estado do Vendedor",
    template="plotly_dark", height=500,
    xaxis_title="Estado", yaxis_title="Taxa de Atraso (%)",
)
fig.show()

## 8. Conclusões

### Principais achados:

1. **Pareto confirmado:** Poucos vendedores concentram a maioria dos atrasos. Ações direcionadas a esse grupo teriam impacto desproporcional na melhoria global.

2. **Atraso impacta diretamente a satisfação:** Vendedores com alta taxa de atraso apresentam notas significativamente piores.

3. **Concentração geográfica:** Alguns estados apresentam taxas de atraso sistematicamente maiores, possivelmente por problemas de infraestrutura logística.

### Recomendações:

- **Programa de monitoramento**: Alertas automáticos para vendedores com taxa de atraso > 15%.
- **Plano de ação progressivo**: Aviso → penalização de ranking → suspensão temporária.
- **Incentivo logístico**: Apoiar vendedores de estados com alta taxa de atraso com soluções de fulfillment regionais.